# PyMISP with MISP 2.5 — workshop

Tested reference: **PyMISP 2.5.34.4**, Python 3.12 and **MISP 2.5.50**. See the [course README](README.md) for setup, REST mapping, permissions, source provenance and validation limits.

**Default execution is offline.** Local event/model creation does not contact MISP. Set `MISP_LIVE=1` outside this notebook to enable the read cells. `MISP_LAB_WRITE=1` additionally enables the separate disposable write exercise. Keep credentials outside notebook source/outputs and use only an authorised training instance. Review each cell before enabling live work.

In [ ]:
import os
from importlib.metadata import version
from pymisp import MISPEvent, MISPAttribute
from examples.models import build_event
from examples.common import Config, TrainingError, connect
from examples.read import one_page, collect_pages

print("PyMISP", version("pymisp"))
LIVE = os.environ.get("MISP_LIVE") == "1"
WRITE_LAB = os.environ.get("MISP_LAB_WRITE") == "1"
event_uuid = os.environ.get("MISP_EVENT_UUID") or None

## 1. Build the model offline

Adapted from the `MISPEvent`, attribute and `domain-ip` examples in the upstream FullOverview notebook. The helper uses unique `.example` domains, `192.0.2.10`, explicit `to_ids=False`, organisation-only distribution and an unpublished draft. These are synthetic teaching observations, not a production blocklist.

In [ ]:
draft = build_event()
attribute = draft.attributes[0]
obj = draft.objects[0]
print(draft.info)
print("Event UUID:", draft.uuid)
print("Top-level attributes:", len(draft.attributes), "objects:", len(draft.objects))
print("Reference target:", obj.references[0].referenced_uuid)
assert obj.references[0].referenced_uuid == attribute.uuid
obj.to_json(strict=True)  # Check local object-template requirements; no server request.
print("Local template check passed.")

## 2. Round-trip JSON; distinguish local edits from persistence

`to_json()` returns a JSON string. `load()` rebuilds a local model; neither method saves to a server. Model/template validation does not establish API permissions, server acceptance or evidential truth.

In [ ]:
encoded = draft.to_json(indent=2)
restored = MISPEvent()
restored.load(encoded)
assert restored.uuid == draft.uuid
assert restored.objects[0].references[0].referenced_uuid == attribute.uuid
restored.attributes[0].comment = "Reviewed locally; not yet persisted."
print("Round-trip preserved identity and references.")

## 3. Optional live connection

Configure `MISP_URL` and `MISP_API_KEY` through your local secret mechanism before starting the kernel. Use trusted HTTPS; `REQUESTS_CA_BUNDLE` may name a private-CA bundle. Only a disposable loopback lab may use HTTP. The configuration helper preserves deployment subpaths with a trailing slash.

The `PyMISP` constructor makes server requests. A read-only key can perform these reads and POST searches, subject to the account's visibility and API role. Do not display the client/configuration or key.

In [ ]:
config = Config.from_env() if LIVE else None
misp = connect(config) if LIVE else None
if LIVE:
    server = misp.misp_instance_version
    if not isinstance(server, dict) or "version" not in server:
        raise TrainingError("Could not read the server version.")
    print("Server:", server["version"])
    print("Recommended PyMISP:", server.get("pymisp_recommended_version", "not supplied"))
else:
    print("Live connection disabled; offline cells remain runnable.")

## 4. One checked page of typed attributes

The query selects domains ending in `.example` and explicitly chooses **attribute scope**. `type_attribute`, `return_format` and `include_event_uuid` map to REST `type`, `returnFormat` and `includeEventUuid`.

For ordinary JSON in this snapshot, omitting `published` and `to_ids` includes both states. Do not use `to_ids=[0,1]` with this PyMISP release: the wrapper rejects it. Set `MISP_EVENT_UUID` to restrict the exercise to a known visible fixture; otherwise this reads one page of visible `.example` domains. This is not a complete export.

In [ ]:
typed_rows = one_page(misp, event_uuid=event_uuid, limit=10, page=1) if LIVE else []
if LIVE:
    print("One page:", len(typed_rows), "attributes; not a complete export.")
    for row in typed_rows:
        print(row.uuid, row.type)
else:
    print("Live attribute read skipped.")

## 5. Optional comparison with the raw REST request

Raw attribute JSON has `response.Attribute`; PyMISP normalises the outer envelope and `pythonify=True` returns typed objects. Compare UUIDs for the **same stable query/page**, not the shape of the JSON wrappers. Concurrent edits can change the result between requests. This cell prints only a count and comparison status.

In [ ]:
if LIVE:
    import requests
    raw_query = {
        "returnFormat": "json", "type": "domain", "value": "%.example",
        "includeEventUuid": True, "includeEventTags": True,
        "order": "Attribute.id ASC", "limit": 10, "page": 1,
    }
    if event_uuid:
        raw_query["eventid"] = event_uuid
    response = requests.post(
        config.url + "attributes/restSearch", json=raw_query,
        headers={"Authorization": config.key, "Accept": "application/json"},
        verify=config.verify, timeout=(5, 60), allow_redirects=False,
    )
    if not 200 <= response.status_code < 300:
        raise TrainingError("Raw read failed; check URL, permissions and response locally.")
    payload = response.json()
    if not isinstance(payload, dict) or "errors" in payload:
        raise TrainingError("Unexpected raw response.")
    envelope = payload.get("response", {})
    if not isinstance(envelope, dict) or not isinstance(envelope.get("Attribute"), list):
        raise TrainingError("Expected raw response.Attribute array.")
    raw_rows = envelope["Attribute"]
    if {r["uuid"] for r in raw_rows} != {r.uuid for r in typed_rows}:
        raise TrainingError("UUID sets differ; inspect query equivalence and concurrent changes.")
    print("Equivalent raw/typed UUID sets:", len(raw_rows))
else:
    print("Raw REST comparison skipped.")

## 6. Optional bounded traversal

For this fixed ordinary JSON query, continue through short pages to an empty page. Role limits can cap the requested size. The helper fails on repeated UUIDs or a maximum-page bound rather than claiming truncated output is complete. It holds this small result in memory, saves no checkpoint and is not a generic sync/export client. Warninglist/decay post-filtering needs a different completion policy.

In [ ]:
if LIVE:
    rows = collect_pages(misp, event_uuid=event_uuid, limit=10, max_pages=20)
    print("Completed bounded query:", len(rows), "attributes.")
else:
    print("Live traversal skipped.")

## 7. Separate opt-in: the disposable write workflow

Use only an isolated training instance and a permitted writer; the operator must make `tlp:clear` and the `domain-ip` template available. With **both** live flags enabled, this cell:

1. Creates its own organisation-only, unpublished synthetic event with attributes, object and reference.
2. Updates one attribute, attaches an event tag and a direct local attribute tag, and submits a synthetic type-0 sighting.
3. Refetches and verifies saved state; reads pages restricted to the new event UUID.
4. Deletes only that confirmed fixture in `finally` and checks its absence.

It never publishes. A timeout can occur after persistence: inspect uncertain writes/cleanup before rerunning. Read [workshop.py](examples/workshop.py) before opting in. A proposal is not proof that an original attribute was edited.

In [ ]:
if LIVE and WRITE_LAB:
    from examples.workshop import run_workshop
    summary = run_workshop(misp)
    print(summary)  # UUID/counts only; the fixture has been deleted.
else:
    print("Write workshop disabled. Requires MISP_LIVE=1 and MISP_LAB_WRITE=1.")

## Discuss and extend

- Why can a full matching event include an IP when the query selected domains?
- Which tags are inherited context, and which are direct/local attachments?
- Why is `last`/`publish_timestamp` different from an attribute `timestamp` window?
- How would you handle role-capped pages, access loss, tombstones and uncertain POST outcomes?
- Why does a local JSON round-trip not prove a server write or an attribution claim?

Examples are adapted from the pinned [upstream FullOverview](https://github.com/MISP/PyMISP/blob/45c906070c287dffeed353069cb3daddceab2ea3/docs/tutorial/FullOverview.ipynb), [Search-FullOverview](https://github.com/MISP/PyMISP/blob/45c906070c287dffeed353069cb3daddceab2ea3/docs/tutorial/Search-FullOverview.ipynb) and the [archived Introduction notebook](../archive/slide-decks/a.7-rest-API-v1/Introduction_to_PyMISP.ipynb). See README for cell provenance and changes. Clear outputs before committing/sharing; keep credentials and private responses out of shared notebooks.